# OmniForge - Google Colab Training

This notebook trains the OmniForge 125M parameter language model from scratch.
It runs each pipeline step sequentially: data collection → cleaning → deduplication → tokenization → training.

## Cell 1: Mount Google Drive
Mount Google Drive to persist checkpoints and datasets across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Clone the OmniForge Repository
Replace `YOUR_REPO_URL` with your actual GitHub repository URL.

In [ ]:
# Replace with your actual repository URL
REPO_URL = "https://github.com/YOUR_USERNAME/omniforge.git"

!git clone {REPO_URL}
%cd omniforge

## Cell 3: Install Dependencies
Install all required Python packages from requirements.txt.

In [ ]:
!pip install -r requirements.txt

## Cell 4: Create Project Directories
Ensure all data and checkpoint directories exist.

In [ ]:
import os
for d in ['data/raw', 'data/clean', 'data/tokenized', 'tokenizer', 'checkpoints', 'logs']:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

## Cell 5: Restore Checkpoint from Google Drive
If a previous checkpoint exists on Google Drive, copy it locally to resume training.

In [ ]:
import glob
import shutil

drive_ckpt_dir = '/content/drive/MyDrive/omniforge_checkpoints/'
local_ckpt_dir = 'checkpoints/'

if os.path.exists(drive_ckpt_dir):
    for ckpt in os.listdir(drive_ckpt_dir):
        shutil.copy(os.path.join(drive_ckpt_dir, ckpt), local_ckpt_dir)
    print(f'Restored checkpoints from Google Drive.')
else:
    os.makedirs(drive_ckpt_dir, exist_ok=True)
    print('No existing checkpoints found. Starting fresh.')

## Cell 6: Download Dataset
Download up to 500,000 Python/JavaScript code documents from Hugging Face's the-stack-dedup.

In [ ]:
!python dataset_collector.py --max-docs 500000

## Cell 7: Clean Dataset
Filter and quality-score the raw dataset.

In [ ]:
!python dataset_cleaner.py

## Cell 8: Deduplicate
Remove exact duplicate documents using SHA256 hashing.

In [ ]:
!python deduplicator.py

## Cell 9: Train Tokenizer
Train a BPE tokenizer with 32,000 vocabulary size on up to 500,000 sample documents.

In [ ]:
!python train_tokenizer.py

## Cell 10: Prepare Dataset
Tokenize all documents, pack into context-length chunks, and split into train/val/test.

In [ ]:
!python prepare_dataset.py

## Cell 11: Train Model
Train the OmniForge 125M parameter transformer from scratch. Training logs are saved to training_log.csv.

In [ ]:
!python train.py

## Cell 12: Backup Checkpoint to Google Drive
Copy the latest checkpoint to Google Drive for persistence.

In [ ]:
import glob
import shutil

local_ckpt_dir = 'checkpoints/'
drive_ckpt_dir = '/content/drive/MyDrive/omniforge_checkpoints/'
os.makedirs(drive_ckpt_dir, exist_ok=True)

checkpoints = sorted(glob.glob(os.path.join(local_ckpt_dir, 'checkpoint_step_*.pt')))
if checkpoints:
    latest = checkpoints[-1]
    shutil.copy(latest, drive_ckpt_dir)
    print(f'Backed up {os.path.basename(latest)} to Google Drive.')
else:
    print('No checkpoints found to backup.')

## Training Complete!

Your OmniForge 125M model has been trained. You can now:
- Run `python inference.py --prompt "def hello():"` for CLI inference
- Run `python server.py` to start the web inference server
- Run `python evaluate.py` to evaluate perplexity and code completion